In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

import openai
client = openai.AsyncOpenAI()

The input dataset is a long-format dataframe, with headings including type (type of test, like ‘uses’ for alternate uses task), src (a dataset id), question (a long version of the prompt that the participant responded to), prompt (a short version of the prompt), response (the participant’s input), id, and language (3-character ISO). For example:

In [ ]:
root_dir = Path('../../data/translation')
data = pd.read_csv(root_dir / '..' / 'ocsai-all.csv')
# normalize some completions data
for [og, fix] in [['friend phone', 'phone'], ['games', 'rain']]:
    data.loc[(data.type == 'completion') & (data.prompt == og), 'prompt'] = fix

for col in ['language', 'prompt', 'question', 'response']:
    data = data.rename(columns={col: f'original_{col}'})
display(data.head(1))
all_langs = data.original_language.unique().tolist()
all_langs


/var/folders/k2/vg86l2h54czgjfzlmd6fkz908bdx0t/T/ipykernel_2531/1799035845.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(root_dir / 'ocsai-all.csv')


,type,src,original_question,original_prompt,original_response,id,target,participant,response_num,original_language,rater_count,rating_std,dupe_control,participant_list,default_split,prompt_split,lang_split,type_split
0,uses,multiaut_arabic1,ما هو الاستخدام المفاجئ لـ علب الصفيح؟,علب الصفيح,آلات حادة,multiaut_arabic1_علب الصفيح-b9c479,2.0,multiaut_arabic1101,6.0,ara,1,NaN,آلات حادة,['multiaut_arabic1101'],train,train,test,train


['ara', 'chi', 'dut', 'eng', 'fre', 'ger', 'heb', 'ita', 'pol', 'rus', 'spa']

In [4]:
from crossling_utils import translate_dataframe

## Step 1 - translate prompts/questions

In [ ]:
translate_cols = ['original_prompt', 'original_question']
# remember to drop_duplicates to avoid redundant translations!!
to_translate = data[['type', 'original_language'] + translate_cols].drop_duplicates().reset_index().drop(columns=['index'])
translations = await translate_dataframe(client, to_translate, translate_cols, all_langs)
translations.to_csv(root_dir / 'ocsai2_translated_prompts_and_questions.csv')

## Step 2: Translate responses

In [5]:
translate_cols = ['original_response']
include_cols = ['original_question']
to_translate = data[['type', 'original_language'] + include_cols + translate_cols].drop_duplicates().reset_index().drop(columns=['index'])
# See code below
#response_translations = await translate_dataframe(client, to_translate, translate_cols, all_langs, include_cols=include_cols)
#.to_csv(root_dir / 'ocsai2_translated_responses.csv')

### Version with intermediately saved batches

In [ ]:
# Set up batch processing parameters
from tqdm.auto import tqdm
batch_size = 100  # Number of rows per batch
total_rows = len(to_translate)
num_batches = (total_rows + batch_size - 1) // batch_size
resume_from_batch = 0

# Create output directory if it doesn't exist
output_dir = root_dir / 'translation_batches'
output_dir.mkdir(exist_ok=True)

for batch_num in tqdm(range(resume_from_batch, num_batches), desc="Outer batches"):
    start_idx = batch_num * batch_size
    end_idx = min((batch_num + 1) * batch_size, total_rows)
    current_batch = to_translate.iloc[start_idx:end_idx]

    batch_translations = await translate_dataframe(
        client, 
        current_batch, 
        translate_cols, 
        all_langs, 
        batch_size=20,
        max_concurrent=20,
        include_cols=include_cols
    )
    
    # Save the batch results
    batch_file = output_dir / f'translated_batch_{batch_num:03d}.csv'
    batch_translations.to_csv(batch_file)

In [ ]:
import textwrap
from ocsai.prompt.utils import strip_backticks
from ocsai.utils import generic_llm
import anthropic
client = anthropic.Anthropic()

In [ ]:
all_files = sorted(output_dir.glob('translated_batch_*.csv'))
final_translations = pd.concat([pd.read_csv(f, index_col=0) for f in list(all_files)])

final_file = root_dir / 'ocsai2_translated_responses.csv'
final_translations.to_csv(final_file)
print(f"Saved final combined translations to {final_file}")

# Display the final results
display(final_translations.sample(5))

# check for errors - goal is <100 (0.1%)
remainder = to_translate.merge(final_translations, how='left')
remainder = remainder[remainder['response_eng'].isna()][to_translate.columns]
remainder.shape

,type,original_language,original_question,original_response,response_ara,response_chi,response_dut,response_eng,response_fre,response_ger,response_heb,response_ita,response_pol,response_rus,response_spa
33,completion,eng,"Complete this sentence in a surprising way: ""M...",the world was ending,أن العالم كان ينتهي,世界要结束了,de wereld ten einde liep,the world was ending,que le monde était en train de se terminer,die Welt zu Ende ging,שהעולם נגמר,che il mondo stava finendo,że świat się kończy,что мир заканчивается,que el mundo se estaba acabando
67,uses,chi,袜子的一个令人惊讶的用途是什么？,水池塞子,سدادة حوض,水池塞子,afvoerstop,sink stopper,bouchon de lavabo,Waschbeckenstöpsel,פקק כיור,tappo per lavandino,zatyczka do zlewu,пробка для раковины,tapón de lavabo
45,uses,pol,Jakie jest zaskakujące zastosowanie dla PUSZKI?,Pudło rezonansowe do wzmocnienia dźwięku np sm...,صندوق رنين لتعزيز الصوت مثل الهاتف الذكي,共鸣箱，用于增强例如智能手机的声音,"Resonantiekast om geluid te versterken, bijvoo...","Resonance box to amplify sound, e.g., from a s...","Boîte de résonance pour amplifier le son, par ...","Resonanzkasten zur Verstärkung des Sounds, z.B...","תיבת תהודה לחיזוק הצליל, למשל של סמארטפון","Cassa di risonanza per amplificare il suono, a...",Pudło rezonansowe do wzmocnienia dźwięku np sm...,"Резонирующий ящик для усиления звука, например...","Caja de resonancia para amplificar el sonido, ..."


(11, 4)

**Description of process.** The translation function was written and run over 100k responses. Afterwards, we looked at 570 responses that failed and updated the parsing code. The two primary issues were quotation marks used inside a value, which messed up the response parsing. This is particularly common in Hebrew. That accounted for about half of the remaining errors. The other half comprised Polish responses where the sentence started with "ich," which means "their" and the model started repeating the word (e.g. 'ich ich ich...'). That seems to be a training error in `gpt-4o-mini`, which we addressed through prompt adjustments. Final rows that needed fixing were translated with `gpt-4o`.

In [ ]:
# for fixing errors, scale up to gpt-4o (non-mini)
#translation_fixes = await translate_dataframe(client, remainder, 
#                                              translate_cols,
#                                              all_langs,
#                                              model='gpt-4o',
#                                              max_concurrent=20,
#                                              batch_size=20,
#                                              include_cols=include_cols)
